In [ ]:
# ----------------------------------------
#  PREVIO: Generación de df_merged
# ----------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.stats import gaussian_kde

In [ ]:
# Cargar datos de captura
captura = pd.read_csv("data/raw/captura-puerto-flota-2019.csv", encoding='latin1')
df_captura = captura.copy()
df_captura["fecha"] = pd.to_datetime(df_captura["fecha"])
df_captura["mes"] = df_captura["fecha"].dt.month
captura_mensual = df_captura.groupby("mes")["captura"].sum().reset_index()

In [ ]:
# Cargar clima mensual
clima = pd.read_csv("data/raw/sst_anomaly_tdf_2019_mensual.csv")
if "fecha" in clima.columns:
    clima["fecha"] = pd.to_datetime(clima["fecha"])
    clima["mes"] = clima["fecha"].dt.month

In [ ]:
assert "anom" in clima.columns, "La columna 'anom' no está presente en clima"

In [ ]:
# Merge final
df_merged = captura_mensual.merge(clima[["mes", "anom"]], on="mes", how="left")
df_merged["captura_log"] = np.log1p(df_merged["captura"])
df_merged["interaccion_temp"] = df_merged["captura"] * df_merged["anom"]
df_merged["anom_abs"] = df_merged["anom"].abs()

In [ ]:
# Regresión lineal
X = df_merged[["anom", "anom_abs", "interaccion_temp"]]
y = df_merged["captura"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(range(len(y_test)), y_test.values, marker='o', label='Real')
plt.plot(range(len(y_pred)), y_pred, marker='x', label='Predicho')
plt.title("Predicción vs Real (Regresión Lineal)")
plt.xlabel("Observaciones")
plt.ylabel("Captura")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Escalado y KDE
scaler = MinMaxScaler()
df_scaled = df_merged.copy()
df_scaled[["anom", "captura"]] = scaler.fit_transform(df_scaled[["anom", "captura"]])

In [ ]:
plt.figure(figsize=(10,5))
sns.kdeplot(data=df_scaled, x="captura", fill=True, label="Captura")
sns.kdeplot(data=df_scaled, x="anom", fill=True, label="Anomalía SST")
plt.title("Estimador de Densidad (KDE) - Datos Escalados")
plt.xlabel("Valor Normalizado")
plt.ylabel("Densidad")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Histograma de densidad personalizado
x = df_scaled["captura"].values[:, np.newaxis]
n_bins = np.linspace(x.min(), x.max(), 6)
plt.figure(figsize=(10, 5))
plt.hist(x, bins=n_bins, density=True, facecolor='red', edgecolor='black', alpha=0.7)
plt.title("Histograma de Densidad - Captura de Merluza Negra (Escalado)")
plt.xlabel("Captura (valor normalizado)")
plt.ylabel("Densidad")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# KDE estandarizado con scipy
scaler_std = StandardScaler()
df_std = df_merged.copy()
df_std[["captura", "anom"]] = scaler_std.fit_transform(df_std[["captura", "anom"]])
sns.kdeplot(data=df_std, x="captura", fill=True, label="Captura (estandarizada)")
plt.title("Densidad de Captura estandarizada")
plt.show()

In [ ]:
captura_array = df_std["captura"].dropna().values
anom_array = df_std["anom"].dropna().values
kde_captura = gaussian_kde(captura_array)
kde_anom = gaussian_kde(anom_array)
x_vals = np.linspace(min(captura_array.min(), anom_array.min()), max(captura_array.max(), anom_array.max()), 200)
plt.figure(figsize=(10, 5))
plt.plot(x_vals, kde_captura(x_vals), label="Captura (estandarizada)", color="skyblue", linewidth=2)
plt.plot(x_vals, kde_anom(x_vals), label="Anomalía SST (estandarizada)", color="orange", linewidth=2)
plt.fill_between(x_vals, kde_captura(x_vals), alpha=0.3, color="skyblue")
plt.fill_between(x_vals, kde_anom(x_vals), alpha=0.3, color="orange")
plt.title("Comparación de Densidad: Captura vs Anomalía SST (Estandarizadas)")
plt.xlabel("Valor estandarizado")
plt.ylabel("Densidad")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest con log-transformación
df_transformed = df_merged.copy()
df_transformed["captura_log"] = np.log1p(df_transformed["captura"])
df_scaled_log = df_transformed.copy()
df_scaled_log[["anom", "captura_log"]] = scaler_std.fit_transform(df_scaled_log[["anom", "captura_log"]])

In [ ]:
X = df_scaled_log[["anom"]]
y = df_scaled_log["captura_log"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42)
modelo_rf.fit(X_train, y_train)
y_pred = modelo_rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred)
r2_rf = r2_score(y_test, y_pred)
print(f"MAE RF: {mae_rf:.2f}")
print(f"R² RF: {r2_rf:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, color='seagreen', edgecolor='k')
plt.plot([-2, 2], [-2, 2], 'k--', lw=2)
plt.xlabel("Captura real (log-transformada, escalada)")
plt.ylabel("Captura predicha (log-transformada, escalada)")
plt.title("Captura Real vs Predicha (Random Forest con Log)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()